In [2]:
import pandas as pd
import numpy as np
import torch
from data_load import load_data

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import ttest_ind, mannwhitneyu
import torch
from transformers import AutoTokenizer, AutoModel
from preprocess import preprocess_data
   # Preprocess data
processed_data = preprocess_data(first_date_to_keep = '2024-11-01',)

print(processed_data.shape)
print(processed_data.head())

(132512, 19)
  date_received                                            product  \
0    2025-09-25  Credit reporting or other personal consumer re...   
1    2025-08-15  Credit reporting or other personal consumer re...   
2    2025-01-20  Money transfer, virtual currency, or money ser...   
3    2025-09-30                        Checking or savings account   
4    2025-07-29                                       Prepaid card   

                    sub-product                                 issue  \
0              Credit reporting  Incorrect information on your report   
1              Credit reporting           Improper use of your report   
2  Domestic (US) money transfer             Other transaction problem   
3              Checking account                   Managing an account   
4  General-purpose prepaid card              Unexpected or other fees   

                                           sub-issue  \
0                Information belongs to someone else   
1  Credit inqui

In [3]:
#make sure its on gpu 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:

# Define embedding models
models_to_test = {
    'mini': 'sentence-transformers/all-miniLM-L6-v2',  # Current model
    'mpnet': 'sentence-transformers/all-mpnet-base-v2',  # Alternative model
    # 'model_c': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
}

def get_embeddings_for_model(texts, model_name, device):
    """Generate embeddings for a specific model"""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model = model.to(device)
    embeddings = []
    batch_size = 32
    
    for i in range(0, len(texts), batch_size):
        if i % (batch_size * 100) == 0:
            print(f'Processing batch {i//batch_size + 1} / {(len(texts) + batch_size - 1)//batch_size}')
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        batch_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.append(batch_emb)
    
    return np.vstack(embeddings)
def get_embeddings_dict(dta, model_name, device):
    """Generate embeddings dictionary for given data and model"""
    narratives = dta['consumer_complaint_narrative'].tolist()
    embeddings = get_embeddings_for_model(narratives, model_name, device)
    embeddings_dict = {
    'complaint_id': dta['complaint_id'].tolist(),
    'product': dta['product'].tolist(),
    'sub-product': dta['sub-product'].tolist(),
    'company': dta['company'].tolist(),
    'narrative': dta['consumer_complaint_narrative'].tolist(),
    'embeddings': embeddings
    }
    return embeddings_dict


#load data 



In [9]:
#embed the narratives
#time each model
import time

start_time = time.time()
mini_embeddings_dict = get_embeddings_dict(processed_data, models_to_test['mini'], device)
print(f"MiniLM embedding time: {time.time() - start_time:.2f} seconds")
print(f"MiniLM embedding time: {(time.time() - start_time)/60:.2f} minutes")

start_time = time.time()
mpnet_embeddings_dict = get_embeddings_dict(processed_data, models_to_test['mpnet'], device)
print(f"MPNet embedding time: {time.time() - start_time:.2f} seconds")  
print(f"MPNet embedding time: {(time.time() - start_time)/60:.2f} minutes")

Processing batch 1 / 4141
Processing batch 101 / 4141
Processing batch 201 / 4141
Processing batch 301 / 4141
Processing batch 401 / 4141
Processing batch 501 / 4141
Processing batch 601 / 4141
Processing batch 701 / 4141
Processing batch 801 / 4141
Processing batch 901 / 4141
Processing batch 1001 / 4141
Processing batch 1101 / 4141
Processing batch 1201 / 4141
Processing batch 1301 / 4141
Processing batch 1401 / 4141
Processing batch 1501 / 4141
Processing batch 1601 / 4141
Processing batch 1701 / 4141
Processing batch 1801 / 4141
Processing batch 1901 / 4141
Processing batch 2001 / 4141
Processing batch 2101 / 4141
Processing batch 2201 / 4141
Processing batch 2301 / 4141
Processing batch 2401 / 4141
Processing batch 2501 / 4141
Processing batch 2601 / 4141
Processing batch 2701 / 4141
Processing batch 2801 / 4141
Processing batch 2901 / 4141
Processing batch 3001 / 4141
Processing batch 3101 / 4141
Processing batch 3201 / 4141
Processing batch 3301 / 4141
Processing batch 3401 / 41

In [11]:
#get the ideal # of clusters by product from prvious best model results
best_model= pd.read_csv('output/best_distance_thresholds.csv')
best_model

,product,best_distance_threshold,num_clusters,silhouette
0,"Money transfer, virtual currency, or money ser...",40,16,0.415
1,Checking or savings account,30,26,0.183
2,Credit card,20,32,0.024
3,Credit reporting or other personal consumer re...,20,36,0.032
4,Debt collection,40,3,0.164
5,Vehicle loan or lease,25,3,0.170
6,Prepaid card,40,3,0.146
7,Mortgage,40,2,0.252
8,"Payday loan, title loan, personal loan, or adv...",20,2,0.235
9,Student loan,10,2,0.343


In [18]:
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict

def evaluate_embeddings(embeddings_dict, n_clusters):
    """Evaluate embeddings using clustering metrics"""
    embeddings = embeddings_dict['embeddings']
    
    # Perform agglomerative clustering
    agg_clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels = agg_clustering.fit_predict(embeddings)
    
    # Calculate metrics
    silhouette = silhouette_score(embeddings, labels)
    davies_bouldin = davies_bouldin_score(embeddings, labels)
    calinski_harabasz = calinski_harabasz_score(embeddings, labels)
    
    return {
        'silhouette': silhouette,  # Higher is better
        'davies_bouldin': davies_bouldin,  # Lower is better
        'calinski_harabasz': calinski_harabasz  # Higher is better
    }

def compare_models_by_product(mini_dict, mpnet_dict, max_samples=5000):
    """Compare models for each product with memory constraints"""
    results = defaultdict(dict)
    # Determine number of clusters per product from best_model
    n_clusters_dict = dict(zip(best_model['product'], best_model['num_clusters']))
    # Get unique products
    products = set(mini_dict['product'])
    
    # Convert product lists to numpy arrays for efficient indexing
    mini_products = np.array(mini_dict['product'])
    mpnet_products = np.array(mpnet_dict['product'])
    
    for product in products:
        print(f"\nProcessing: {product}")
        
        # Filter by product using numpy boolean indexing
        mini_indices = np.where(mini_products == product)[0]
        mpnet_indices = np.where(mpnet_products == product)[0]
        
        # Skip if too few samples
        if len(mini_indices) < 10:
            print(f"  Skipped: Too few samples ({len(mini_indices)})")
            continue
        
        # Sample if too many data points to avoid memory issues
        if len(mini_indices) > max_samples:
            print(f"  Sampling {max_samples} from {len(mini_indices)} samples")
            sample_indices = np.random.choice(mini_indices, max_samples, replace=False)
            mini_product_emb = mini_dict['embeddings'][sample_indices]
            mpnet_product_emb = mpnet_dict['embeddings'][sample_indices]
        else:
            mini_product_emb = mini_dict['embeddings'][mini_indices]
            mpnet_product_emb = mpnet_dict['embeddings'][mpnet_indices]
        
        # Create filtered dicts
        mini_filtered = {'embeddings': mini_product_emb}
        mpnet_filtered = {'embeddings': mpnet_product_emb}
        
        # Evaluate both models
        n_clusters = int(n_clusters_dict.get(product, best_model['num_clusters'].mean()))
        
        # Make sure n_clusters doesn't exceed sample size
        n_clusters = min(n_clusters, len(mini_product_emb) - 1, 50)  # Cap at 50 clusters
        
        print(f"  Clustering with {n_clusters} clusters on {len(mini_product_emb)} samples")
        
        results[product]['mini'] = evaluate_embeddings(mini_filtered, n_clusters)
        results[product]['mpnet'] = evaluate_embeddings(mpnet_filtered, n_clusters)
        results[product]['sample_size'] = len(mini_indices)
        results[product]['sampled'] = len(mini_indices) > max_samples
    
    return results


# Run comparison with memory constraints
np.random.seed(42)  # For reproducibility
comparison_results = compare_models_by_product(mini_embeddings_dict, mpnet_embeddings_dict, max_samples=5000)



# Run comparison with memory constraints
np.random.seed(42)  # For reproducibility
comparison_results = compare_models_by_product(mini_embeddings_dict, mpnet_embeddings_dict, max_samples=5000)


# Run comparison
comparison_results = compare_models_by_product(mini_embeddings_dict, mpnet_embeddings_dict)

summary_results = pd.DataFrame()

# Display results
for product, metrics in comparison_results.items():
    print(f"\n{'='*60}")
    print(f"Product: {product}")
    print(f"Sample Size: {metrics['sample_size']}")
    print(f"\nMiniLM:")
    print(f"  Silhouette: {metrics['mini']['silhouette']:.4f}")
    print(f"  Davies-Bouldin: {metrics['mini']['davies_bouldin']:.4f}")
    print(f"  Calinski-Harabasz: {metrics['mini']['calinski_harabasz']:.2f}")
    print(f"\nMPNet:")
    print(f"  Silhouette: {metrics['mpnet']['silhouette']:.4f}")
    print(f"  Davies-Bouldin: {metrics['mpnet']['davies_bouldin']:.4f}")
    print(f"  Calinski-Harabasz: {metrics['mpnet']['calinski_harabasz']:.2f}")
    
    # Determine winner
    mini_score = (metrics['mini']['silhouette'] - 
                  metrics['mini']['davies_bouldin']/10 + 
                  metrics['mini']['calinski_harabasz']/1000)
    mpnet_score = (metrics['mpnet']['silhouette'] - 
                   metrics['mpnet']['davies_bouldin']/10 + 
                   metrics['mpnet']['calinski_harabasz']/1000)
    
    winner = 'MiniLM' if mini_score > mpnet_score else 'MPNet'
    print(f"\nWinner: {winner}")
    summary_results = pd.concat([summary_results, pd.DataFrame({
        'product': [product],

        'mini_silhouette': [metrics['mini']['silhouette']],
        'mini_davies_bouldin': [metrics['mini']['davies_bouldin']],
        'mini_calinski_harabasz': [metrics['mini']['calinski_harabasz']],
        'mpnet_silhouette': [metrics['mpnet']['silhouette']],
        'mpnet_davies_bouldin': [metrics['mpnet']['davies_bouldin']],
        'mpnet_calinski_harabasz': [metrics['mpnet']['calinski_harabasz']],
        'winner': [winner]
    })], ignore_index=True)
summary_results.to_csv('output/model_comparison_summary.csv', index=False)


Processing: Debt collection
  Clustering with 3 clusters on 4404 samples

Processing: Mortgage
  Clustering with 2 clusters on 1243 samples

Processing: Debt or credit management
  Clustering with 2 clusters on 205 samples

Processing: Prepaid card
  Clustering with 3 clusters on 1751 samples

Processing: Student loan
  Clustering with 2 clusters on 36 samples

Processing: Checking or savings account
  Sampling 5000 from 31893 samples
  Clustering with 26 clusters on 5000 samples

Processing: Payday loan, title loan, personal loan, or advance loan
  Clustering with 2 clusters on 415 samples

Processing: Money transfer, virtual currency, or money service
  Sampling 5000 from 54312 samples
  Clustering with 16 clusters on 5000 samples

Processing: Credit reporting or other personal consumer reports
  Sampling 5000 from 15246 samples
  Clustering with 36 clusters on 5000 samples

Processing: Credit card
  Sampling 5000 from 21338 samples
  Clustering with 32 clusters on 5000 samples

Pro